In [1]:
!pip install pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.9 MB/s eta 0:00:00


In [2]:
from pypdf import PdfReader

reader = PdfReader("FIA_doc.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:1000])   # preview first part

SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) 
 
 
 
 
0 
 
E 
 
  
E1 2026 Formula 1 Regulations: Financial Regulations (Power Unit Manufacturers) 
©2025 Fédération Internationale de l’Automobile 
10 December 2025 
Issue 03  
SECTION E: FINANCIAL REGULATIONS  
(POWER UNIT MANUFACTURERS) 
 
Version:   Issue 03 
Status:    PUBLISHED 
Date:    10/12/2025 
WMSC approval date:  10/12/2025 
 
CONVENTION: 
Black Text: Text unchanged from FIA 2026 F1 Regulations – Section E [Financial Regulations – Power Unit 
Manufacturers] – Iss 2 – 25-10-16 
[Pink Text]: Changes relative to FIA 2026 F1 Regulations – Section E [Financial Regulations – Power Unit 
Manufacturers] – Issue 02 
[Red Text]: Information on applicable Governance and relevant Advisory Committee 
[Orange Text]: Reference information on relevant FIA F1 Document(s) 
[Green Text]: Comments / explanations / indication of further work: non-binding and non-regulatory 
 
CONTENTS: 
ARTICLE E1: GENERAL PRINCIPLES 3 
E1.1 Scope 

In [3]:
len(text)

151668

In [4]:
chunks=[]
L=800
O=120
S=L-O
for i in range(0,len(text),S):
  chunks.append(text[i*S:i*S+L])

In [5]:
len(chunks)

224

In [6]:
!pip install sentence-transformers

In [ ]:
!pip install huggingface-hub

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
type(embeddings)

numpy.ndarray

In [10]:
import numpy as np
np.shape(embeddings)

(224, 384)

In [11]:
vectors=embeddings.astype('float32')

In [12]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.2 MB/s eta 0:00:00


In [13]:
import faiss

In [14]:
index=faiss.IndexFlatL2(vectors.shape[1])
index.add(vectors)


In [15]:
type(index)

faiss.swigfaiss_avx2.IndexFlatL2

In [16]:
index.reconstruct(0)

array([-7.20193237e-03,  2.98539512e-02, -3.08736768e-02,  2.98446789e-02,
        6.66512474e-02,  6.62236065e-02, -5.87106645e-02,  5.65551780e-02,
       -2.99744948e-04, -3.99920493e-02,  7.97727425e-03,  2.77699996e-03,
        5.46210120e-03,  8.79511982e-03, -1.69275589e-02, -3.84315755e-03,
       -3.41354986e-03, -4.62260582e-02, -6.00969307e-02,  1.72729716e-02,
        1.02658734e-01, -2.40950435e-02, -1.23643978e-02,  3.72202359e-02,
       -5.95577583e-02, -3.60001400e-02, -1.38692111e-02,  1.17092997e-01,
       -2.56400146e-02, -6.55737072e-02, -6.05813712e-02,  3.67576890e-02,
        4.55823652e-02, -3.46448831e-02,  3.18507552e-02, -4.82384451e-02,
        5.06780073e-02, -1.21430568e-02,  1.23604164e-02, -6.70782402e-02,
       -2.90981345e-02, -1.23269990e-01, -3.90604027e-02,  1.02166291e-02,
        3.35456841e-02,  2.14507710e-02,  2.67885625e-02, -1.51028745e-02,
       -2.68008411e-02, -1.90284778e-03,  1.94076151e-02, -2.56384024e-03,
       -4.46644612e-02, -

In [20]:
from google.colab import ai

while True:
  query=input("Enter your Query:")
  query_vec=model.encode([query]).astype('float32')
  k=3
  D,I=index.search(query_vec,k)
  results=[chunks[i] for i in I[0]]
  print(results)
  prompt=f"""
        Answer only using given context.
        If not found,say'I don't know'

        Context:{results}
        Query:{query}
        Answer:
        """
  response = ai.generate_text(prompt)
  print(response)


Enter your Query:What are the new Power Units Manufacturers supposed to take care of?
['SECTION E: FINANCIAL REGULATIONS (POWER UNIT MANUFACTURERS) \n \n \n \n \n0 \n \nE \n \n  \nE1 2026 Formula 1 Regulations: Financial Regulations (Power Unit Manufacturers) \n©2025 Fédération Internationale de l’Automobile \n10 December 2025 \nIssue 03  \nSECTION E: FINANCIAL REGULATIONS  \n(POWER UNIT MANUFACTURERS) \n \nVersion:   Issue 03 \nStatus:    PUBLISHED \nDate:    10/12/2025 \nWMSC approval date:  10/12/2025 \n \nCONVENTION: \nBlack Text: Text unchanged from FIA 2026 F1 Regulations – Section E [Financial Regulations – Power Unit \nManufacturers] – Iss 2 – 25-10-16 \n[Pink Text]: Changes relative to FIA 2026 F1 Regulations – Section E [Financial Regulations – Power Unit \nManufacturers] – Issue 02 \n[Red Text]: Information on applicable Governance and relevant Advisory Committee \n[Orange Text]: Reference in', '', '']
I don't know
Enter your Query:when will the power unit rules come into ac

KeyboardInterrupt: Interrupted by user